In [4]:
# ============================================================
#   GLD/SLV MEAN REVERSION — COMPREHENSIVE EXPERIMENT FRAMEWORK
#   Baseline vs. Looser Short Filter vs. Full Long-Short Leg
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np

# ── 1. GLOBAL PARAMETERS & DATA FETCHING ────────────────────

TICKER_A         = "GLD"
TICKER_B         = "SLV"
START            = "2024-01-01"
END              = "2026-01-01"

WINDOW_LONG      = 252          
HALF_LIFE        = 27.85        
WINDOW_SHORT     = int(HALF_LIFE * 4)   # ~111 days

EXIT_LEVEL       = 0.0         
STOP_LOSS        = -3       
MAX_HOLD_DAYS    = int(HALF_LIFE * 4)   
INITIAL_CAPITAL  = 100.0

print("📥 Downloading data...")
raw = yf.download([TICKER_A, TICKER_B], start=START, end=END, auto_adjust=True)
close_df = raw["Close"][[TICKER_A, TICKER_B]].dropna().copy()
close_df["Ratio"] = close_df[TICKER_A] / close_df[TICKER_B]

# Compute Rolling Statistics
close_df["Mean_Long"]  = close_df["Ratio"].rolling(WINDOW_LONG).mean()
close_df["Std_Long"]   = close_df["Ratio"].rolling(WINDOW_LONG).std()
close_df["Z_Long"]     = (close_df["Ratio"] - close_df["Mean_Long"]) / close_df["Std_Long"]

close_df["Mean_Short"] = close_df["Ratio"].rolling(WINDOW_SHORT).mean()
close_df["Std_Short"]  = close_df["Ratio"].rolling(WINDOW_SHORT).std()
close_df["Z_Short"]    = (close_df["Ratio"] - close_df["Mean_Short"]) / close_df["Std_Short"]

close_df.dropna(inplace=True)

# ── 2. BACKTEST ENGINE ───────────────────────────────────────

def run_backtest(df, name, entry_long_z, entry_short_z, allow_shorts=False):
    """
    Runs a robust mean reversion vector/state simulation handling stops and regime shifts.
    """
    dates = df.index.values
    ratios = df["Ratio"].values
    z_long_arr = df["Z_Long"].values
    z_short_arr = df["Z_Short"].values
    
    position = 0  # 0: Cash, 1: Long Ratio, -1: Short Ratio
    entry_idx = None
    trade_records = []
    
    for i in range(len(df)):
        date = dates[i]
        ratio = ratios[i]
        z_long = z_long_arr[i]
        z_short = z_short_arr[i]
        
        # --- STATE: OUT OF MARKET ---
        if position == 0:
            # Long Check (Gold cheap relative to Silver)
            if z_long < entry_long_z and z_short < entry_short_z:
                position = 1
                entry_idx = i
            # Short Check (Silver cheap relative to Gold)
            elif allow_shorts and (z_long > abs(entry_long_z) and z_short > abs(entry_short_z)):
                position = -1
                entry_idx = i
                
        # --- STATE: LONG RATIO ---
        elif position == 1:
            days_held = (pd.Timestamp(date) - pd.Timestamp(dates[entry_idx])).days
            
            # Check exit conditions
            if z_long >= EXIT_LEVEL:
                trade_records.append((dates[entry_idx], date, ratios[entry_idx], ratio, 1, "PROFIT", days_held))
                position = 0
            elif z_long <= STOP_LOSS:
                trade_records.append((dates[entry_idx], date, ratios[entry_idx], ratio, 1, "STOP_LOSS", days_held))
                position = 0
            elif days_held >= MAX_HOLD_DAYS:
                trade_records.append((dates[entry_idx], date, ratios[entry_idx], ratio, 1, "TIME_STOP", days_held))
                position = 0
                
        # --- STATE: SHORT RATIO ---
        elif position == -1:
            days_held = (pd.Timestamp(date) - pd.Timestamp(dates[entry_idx])).days
            
            # Check exit conditions
            if z_long <= EXIT_LEVEL:
                trade_records.append((dates[entry_idx], date, ratios[entry_idx], ratio, -1, "PROFIT", days_held))
                position = 0
            elif z_long >= abs(STOP_LOSS):
                trade_records.append((dates[entry_idx], date, ratios[entry_idx], ratio, -1, "STOP_LOSS", days_held))
                position = 0
            elif days_held >= MAX_HOLD_DAYS:
                trade_records.append((dates[entry_idx], date, ratios[entry_idx], ratio, -1, "TIME_STOP", days_held))
                position = 0

    # Clean up hanging open positions at the end of the window
    if position != 0:
        days_held = (pd.Timestamp(dates[-1]) - pd.Timestamp(dates[entry_idx])).days
        trade_records.append((dates[entry_idx], dates[-1], ratios[entry_idx], ratios[-1], position, "FORCE_CLOSE", days_held))

    # Calculate Performance Specs
    if not trade_records:
        return {"Name": name, "Trades": 0, "Win Rate": 0, "Expectancy": 0, "Total Return": 0}
        
    t_df = pd.DataFrame(trade_records, columns=["Entry_Date", "Exit_Date", "Entry_Ratio", "Exit_Ratio", "Direction", "Exit_Type", "Hold_Days"])
    
    # Calculate exact return vectors based on directionality
    t_df["Return"] = np.where(t_df["Direction"] == 1, 
                              (t_df["Exit_Ratio"] / t_df["Entry_Ratio"]) - 1, 
                              (t_df["Entry_Ratio"] / t_df["Exit_Ratio"]) - 1)
    
    wins = t_df[t_df["Return"] > 0]
    losses = t_df[t_df["Return"] <= 0]
    
    win_rate = (len(wins) / len(t_df)) * 100
    expectancy = t_df["Return"].mean() * 100
    total_return = ((1 + t_df["Return"]).prod() - 1) * 100
    profit_factor = abs(wins["Return"].sum() / losses["Return"].sum()) if len(losses) > 0 else float("inf")
    avg_hold = t_df["Hold_Days"].mean()
    
    return {
        "Name": name,
        "Trades": len(t_df),
        "Profits": len(t_df[t_df["Exit_Type"] == "PROFIT"]),
        "Stops": len(t_df[t_df["Exit_Type"] == "STOP_LOSS"]),
        "Time Stops": len(t_df[t_df["Exit_Type"] == "TIME_STOP"]),
        "Win Rate": f"{win_rate:.1f}%",
        "Expectancy": f"{expectancy:+.2f}%",
        "Profit Factor": f"{profit_factor:.2f}x",
        "Avg Hold": f"{avg_hold:.1f}",
        "Total Return": f"{total_return:+.1f}%"
    }

# ── 3. EXECUTE PARAMETER PIPELINES ─────────────────────────

results = []

# Baseline Configuration (Tight Long-Only Filters)
results.append(run_backtest(close_df, "Baseline (Strict Both)", -2.0, -2.0, allow_shorts=False))

# Experiment A (Asymmetric, Looser Current Regime Window Filter)
results.append(run_backtest(close_df, "Exp A (Looser Short Z)", -2.0, -1.5, allow_shorts=False))

# Experiment B (Symmetric Continuous Long & Short Regime)
results.append(run_backtest(close_df, "Exp B (Full Long-Short)", -2.0, -2.0, allow_shorts=True))

# Combined Matrix Optimization (Looser + Shorts)
results.append(run_backtest(close_df, "Exp C (Looser + L/S Matrix)", -2.0, -1.5, allow_shorts=True))

# ── 4. RESTRUCTURING DIAGNOSTIC SUMMARY MATRIX ─────────────

summary_df = pd.DataFrame(results)

print("=" * 85)
print("                    STRATEGY PERFORMANCE DIAGNOSTIC MATRIX")
print("=" * 85)
print(f"{'Strategy Variant':<26} {'Trades':<8} {'Wins/Stops/Time':<18} {'Win %':<8} {'Expectancy':<12} {'PF':<6} {'Tot Ret':<10}")
print("-" * 85)
for res in results:
    wst = f"{res['Profits']}/{res['Stops']}/{res['Time Stops']}"
    print(f"{res['Name']:<26} "
          f"{res['Trades']:<8} "
          f"{wst:<18} "
          f"{res['Win Rate']:<8} "
          f"{res['Expectancy']:<12} "
          f"{res['Profit Factor']:<6} "
          f"{res['Total Return']:<10}")
print("=" * 85)

📥 Downloading data...


[*********************100%***********************]  2 of 2 completed

                    STRATEGY PERFORMANCE DIAGNOSTIC MATRIX
Strategy Variant           Trades   Wins/Stops/Time    Win %    Expectancy   PF     Tot Ret   
-------------------------------------------------------------------------------------
Baseline (Strict Both)     9        0/8/0              0.0%     -3.56%       0.00x  -28.0%    
Exp A (Looser Short Z)     9        0/8/0              0.0%     -3.56%       0.00x  -28.0%    
Exp B (Full Long-Short)    15       1/13/0             6.7%     -1.48%       0.47x  -22.0%    
Exp C (Looser + L/S Matrix) 15       1/13/0             6.7%     -1.48%       0.47x  -22.0%    
